### Coleta de dados de Temperatura

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos:
<pre>
- Temperatura   -> variável 2m_temperature, retorna a temperatura em Kelvin, será necessário uma conversão (subtrair -273,15)
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil




In [ ]:
import cdsapi
import sys, os
import xarray as xr
import dask.dataframe as dd
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)
print(DATA_PATH_ROOT)

In [ ]:
def get_data_from_parquet(year):
    path = r"C:\Marco Conti\Projetos\Dados\ERA5-temperaturas\{0}\ERA5_temperatura.parquet".format(year)
    df_temp = spark.read.parquet(path)
    return df_temp


def write_data_csv(df_temperatura, write_path, file_name):
    
    df_temperatura.toPandas().to_csv(f"{write_path}\{file_name}", index=False)

In [ ]:
df_ = get_data_from_parquet(2020)
df_.printSchema()
df_.show(10,False)

# data_medicao,latitude,longitude,indicador,valor,unidade_medida

In [ ]:
year = 2020
csv_path      = r"{DATA_PATH_ROOT}\ERA5-temperaturas\arquivos_CSV".format(DATA_PATH_ROOT = DATA_PATH_ROOT)
csv_file_name = f"ERA5_t2m_{year}.csv"
write_data_csv(df_, csv_path, csv_file_name)

In [ ]:

# # Converte os dados de temperatura para um Spark Dataframe
# ret_download = "ea1aa17d3130f15e9827810d611337cb.nc"
# df_temperatura       = convert_t2m_dataset_to_spark_dataframe(PROJECT_PATH, ret_download)

# # Converte a temperatura de Kelsin para Celsius e adiciona coluna de unidade de medida
# df_temperatura_final = transform_data(df_temperatura)

# # Escreve os dados em formato parquet
# write_data(df_temperatura_final, DATA_PATH_ROOT, 2026)

In [12]:
from datetime import datetime 

# years_process = [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

years_process = [2021, 2022, 2023, 2024, 2025, 2026]

for year in years_process:
    start = datetime(2026, 7, 29).now()
    # print("Start download - year : ",year, " - ", start, end="" )
    print("Start transform - year : ",year, " - ", start, end="" )

    df_ = get_data_from_parquet(year)

    csv_path      = r"{DATA_PATH_ROOT}\ERA5-temperaturas\arquivos_CSV".format(DATA_PATH_ROOT = DATA_PATH_ROOT)
    csv_file_name = f"ERA5_t2m_{year}.csv"
    write_data_csv(df_, csv_path, csv_file_name)

    finish = datetime(2026, 7, 29).now()

    # print(f" - Download completed: {ret_download} - {finish} - {(finish - start)} \n")
    print(f" - Data transform completed: {csv_file_name} - {finish} - {(finish - start)} \n")
        



Start transform - year :  2021  -  2026-08-10 08:56:34.935050 - Data transform completed: ERA5_t2m_2021.csv - 2026-08-10 08:57:03.334770 - 0:00:28.399720 

Start transform - year :  2022  -  2026-08-10 08:57:03.339541 - Data transform completed: ERA5_t2m_2022.csv - 2026-08-10 08:57:30.727463 - 0:00:27.387922 

Start transform - year :  2023  -  2026-08-10 08:57:30.745162 - Data transform completed: ERA5_t2m_2023.csv - 2026-08-10 08:57:58.643373 - 0:00:27.898211 

Start transform - year :  2024  -  2026-08-10 08:57:58.643373 - Data transform completed: ERA5_t2m_2024.csv - 2026-08-10 08:58:30.071137 - 0:00:31.427764 

Start transform - year :  2025  -  2026-08-10 08:58:30.071137 - Data transform completed: ERA5_t2m_2025.csv - 2026-08-10 08:59:14.627410 - 0:00:44.556273 

Start transform - year :  2026  -  2026-08-10 08:59:14.717839 - Data transform completed: ERA5_t2m_2026.csv - 2026-08-10 08:59:35.077570 - 0:00:20.359731 



In [ ]:
# df_temperatura_final.printSchema()
df_temperatura_final.show(10,False)

Converte os dados baixados do ERA5, que estão em formato NetCDF, para um dataset (xarray.core.dataset.Dataset)

In [ ]:
ret_download = "b80f05d1220d008d67fbe12abee7958d.nc" # dados de 2023
# ret_download = "d03a86fd89e8c00cdf3bea8d8fb499f.nc" # dados de 2024
# ret_download = "507e00f8fa0894949bc892e52ab7b3b7.nc" # dados de 2025
path = f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{ret_download}"
print(path)

Renomeia nome de colunas e converte o valor da Temperatura recebida do ERA5 está em Kelvin, para converter para Celsius, subtrair 273.15

In [ ]:
df_temperatura.filter("latitude = -34.0 and longitude = -67.0").orderBy("t2m").show(10,False)

In [ ]:
drop_cols = ["valid_time", "t2m", "number"]

df_temperatura_final = \
    (df_temperatura
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("temperatura") 
                     ,"valor": (F.col("t2m") - F.lit(273.15)).cast("double")
                     ,"unidade_medida": F.lit("celsius")})
         .drop(*drop_cols)
    )

df_temperatura_final = \
    (df_temperatura_final
        .select("data_medicao"
               ,"latitude"
               ,"longitude"
               ,"indicador"
               ,"valor"
               ,"unidade_medida"))

df_temperatura_final.printSchema()

df_temperatura_final.show()

In [ ]:
# df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)

df_temperatura_final.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\mais_einstein\\dados\\ERA5-temperaturas\\2025\\ERA5_temperatura.parquet")
